# Rules 模块教程

本教程详细介绍 open-xquant 的 **Rules 模块**——策略管道中的风控与退出层。

在新架构中，**入场逻辑** 由 Signal + PortfolioOptimizer 完成，**仓位管理** 由 PortfolioOptimizer（EqualWeight、RiskParity、Kelly）自动处理。Rules 专注于 **风控和条件退出**：

```
Strategy 定义:
  - name, universe, signals, portfolio (PortfolioOptimizer)

Engine.run(strategy, market, broker, rules=[...], start=..., end=...)

Rules 的职责:
  - 止损 / 止盈 / 追踪止损 → 条件退出
  - 最大回撤 / 单日亏损   → 组合级熔断
  - 死叉退出              → 信号退出
```

**关键变化**：
- 不再有 EntryRule、SizedEntryRule、FullPositionEntryRule、TargetValueEntryRule
- 不再有 RebalanceRule — 再平衡由 PortfolioOptimizer 每个 bar 自动完成
- 不再有 sizing 函数（clip_to_max_position、clip_to_pct_equity 等）
- 所有 Rule 返回 `RuleResult`（包含 weights/constraints/target_positions/hold/reason）
- Rules 作为列表传给 `Engine.run()`，而非写在 Strategy 定义中

本教程通过 5 个独立章节，演示每种 Rule 的 API 和组合用法。

## 0. 准备工作

### 安装

```bash
pip install open-xquant[yfinance]
```

### 下载数据

本教程使用 AAPL 2023~2024 年的历史数据：

In [ ]:
from oxq.data import YFinanceDownloader

downloader = YFinanceDownloader()
downloader.download("AAPL", start="2023-01-01", end="2024-12-31")
print("数据下载完成")

### 公共配置

所有章节共享同一套 SMA 金叉/死叉信号 + EqualWeightOptimizer 作为基础管道。后续章节只替换 Rule 部分（传给 `Engine.run()` 的 `rules` 参数），保持 Strategy 定义不变，以便公平对比不同 Rule 的效果。

In [ ]:
from decimal import Decimal

from oxq.core import Engine, Strategy
from oxq.data import LocalMarketDataProvider
from oxq.indicators import SMA
from oxq.portfolio.optimizers import EqualWeightOptimizer
from oxq.signals import Crossover
from oxq.trade import SimBroker
from oxq.universe import StaticUniverse

SYMBOL = "AAPL"
START = "2023-01-01"
END = "2024-12-31"
INITIAL_CASH = 100_000.0

# 所有章节共享的 Strategy 定义
base_strategy = Strategy(
    name="sma_cross",
    universe=StaticUniverse((SYMBOL,)),
    signals={
        "golden_cross": (Crossover(), {"fast": "sma_10", "slow": "sma_50"}),
    },
    portfolio=EqualWeightOptimizer(),
)

# Crossover 信号需要 SMA 指标，通过 required_indicators 自动注册


def run(strategy, rules=None, **broker_kwargs):
    """运行策略并返回结果的快捷函数。"""
    broker = SimBroker(**broker_kwargs)
    return Engine().run(
        strategy,
        market=LocalMarketDataProvider(),
        broker=broker,
        rules=rules or [],
        start=START, end=END,
        initial_cash=INITIAL_CASH,
    )


def compare(results: dict, extra_rows=None):
    """打印多策略对比表。"""
    header = f"{'':>14}" + "".join(f"{label:>16}" for label in results)
    print(header)
    print("-" * len(header))
    rows = [
        ("总收益率", lambda r: f"{r.total_return():.2%}"),
        ("Sharpe", lambda r: f"{r.sharpe_ratio():.2f}"),
        ("最大回撤", lambda r: f"{r.max_drawdown():.2%}"),
        ("交易次数", lambda r: f"{len(r.trades)}"),
        ("期末资产", lambda r: f"{r.equity_curve[-1][1]:,.0f}"),
    ]
    if extra_rows:
        rows.extend(extra_rows)
    for name, fn in rows:
        vals = "".join(f"{fn(r):>16}" for r in results.values())
        print(f"{name:>14}{vals}")


def show_trades(result, max_rows=10):
    """打印交易记录。"""
    if not result.trades:
        print("  (无交易)")
        return
    print(f"  {'日期':<28} {'方向':>4} {'数量':>6} {'订单类型':<14} {'成交价':>10} {'手续费':>8}")
    print("  " + "-" * 76)
    for fill in result.trades[:max_rows]:
        o = fill.order
        print(
            f"  {fill.filled_at:<28} {o.side:>4} {o.shares:>6} "
            f"{o.order_type:<14} {fill.filled_price:>10.2f} {fill.fee:>8.2f}"
        )
    if len(result.trades) > max_rows:
        print(f"  ... 共 {len(result.trades)} 笔，仅显示前 {max_rows} 笔")


print(f"公共配置: {SYMBOL}, {START}~{END}, 初始资金 {INITIAL_CASH:,.0f}")

---
## 1. PortfolioOptimizer — 入场与仓位管理（取代 Entry Rules）

在新架构中，**入场和仓位管理不再由 Rule 负责**。Strategy 通过 `portfolio` 字段指定一个 `PortfolioOptimizer`，Engine 每个 bar 调用 `optimizer.optimize()` 获取目标权重，自动计算目标持仓并生成调仓订单。

这意味着：
- **入场逻辑** = Signal 产生信号 + Optimizer 计算权重 → Engine 自动建仓
- **仓位大小** = Optimizer 的权重分配决定
- **再平衡** = Optimizer 每个 bar 重新计算权重，Engine 自动调仓

open-xquant 提供 3 种 PortfolioOptimizer：

| Optimizer | 权重逻辑 | 适用场景 |
|-----------|----------|----------|
| `EqualWeightOptimizer` | 所有标的等权 | 快速验证、分散投资 |
| `RiskParityOptimizer` | 按波动率倒数加权 | 风险均衡配置 |
| `KellyOptimizer` | Kelly 公式计算最优仓位 | 有历史胜率数据时 |

### 1.1 EqualWeightOptimizer — 等权配置

最简单的 Optimizer。将资金平均分配给所有有信号的标的。

单标的时权重为 1.0（全仓），双标的时各 0.5，以此类推。

In [ ]:
import pandas as pd
from oxq.portfolio.optimizers import EqualWeightOptimizer, RiskParityOptimizer, KellyOptimizer

# EqualWeightOptimizer: 等权分配
eq_opt = EqualWeightOptimizer()

# 单标的 → 权重 1.0
weights_1 = eq_opt.optimize(
    signals={"AAPL": pd.DataFrame()},
    indicators={"AAPL": pd.DataFrame()},
)
print(f"单标的权重: {weights_1}")

# 双标的 → 各 0.5
weights_2 = eq_opt.optimize(
    signals={"AAPL": pd.DataFrame(), "MSFT": pd.DataFrame()},
    indicators={"AAPL": pd.DataFrame(), "MSFT": pd.DataFrame()},
)
print(f"双标的权重: {weights_2}")

# 无信号 → 全部现金
weights_0 = eq_opt.optimize(signals={}, indicators={})
print(f"无信号权重: {weights_0}")

### 1.2 RiskParityOptimizer — 风险平价

按波动率的倒数分配权重。波动率低的标的获得更高权重，使每个标的对组合风险的贡献大致相等。

需要 indicators 中包含 `volatility` 列（可通过自定义指标计算）。

In [ ]:
# RiskParityOptimizer: 波动率倒数加权
rp_opt = RiskParityOptimizer(volatility_col="volatility")

# AAPL 波动率 0.2, MSFT 波动率 0.4 → AAPL 权重更高
indicators = {
    "AAPL": pd.DataFrame({"volatility": [0.2]}),
    "MSFT": pd.DataFrame({"volatility": [0.4]}),
}
weights = rp_opt.optimize(
    signals={"AAPL": pd.DataFrame(), "MSFT": pd.DataFrame()},
    indicators=indicators,
)
print(f"RiskParity 权重: {weights}")
print(f"  AAPL (低波动) = {weights['AAPL']:.2%}")
print(f"  MSFT (高波动) = {weights['MSFT']:.2%}")
print(f"  AAPL 权重是 MSFT 的 {weights['AAPL']/weights['MSFT']:.1f} 倍 (因为波动率是 MSFT 的一半)")

### 1.3 KellyOptimizer — Kelly 准则

根据 Kelly 公式计算最优仓位比例：`f* = win_rate - (1 - win_rate) / payoff_ratio`

需要 indicators 中包含 `win_rate`、`avg_win`、`avg_loss` 列。支持 `fraction` 参数缩放 Kelly 比例（fractional Kelly）。

In [ ]:
# KellyOptimizer: Kelly 准则仓位
kelly_opt = KellyOptimizer(fraction=0.5)  # Half-Kelly (更保守)

indicators_kelly = {
    "AAPL": pd.DataFrame({"win_rate": [0.55], "avg_win": [0.08], "avg_loss": [0.04]}),
    "MSFT": pd.DataFrame({"win_rate": [0.50], "avg_win": [0.10], "avg_loss": [0.05]}),
}
weights = kelly_opt.optimize(
    signals={"AAPL": pd.DataFrame(), "MSFT": pd.DataFrame()},
    indicators=indicators_kelly,
)
print(f"KellyOptimizer (fraction=0.5) 权重:")
for sym, w in weights.items():
    print(f"  {sym}: {w:.2%}")

# 全 Kelly (更激进)
kelly_full = KellyOptimizer(fraction=1.0)
weights_full = kelly_full.optimize(
    signals={"AAPL": pd.DataFrame(), "MSFT": pd.DataFrame()},
    indicators=indicators_kelly,
)
print(f"\nKellyOptimizer (fraction=1.0) 权重:")
for sym, w in weights_full.items():
    print(f"  {sym}: {w:.2%}")

### 1.4 回测对比：不同 Optimizer 的效果

将三种 Optimizer 放在同一套信号管道中回测：

In [ ]:
opt_results = {}
for label, optimizer in [
    ("EqualWeight", EqualWeightOptimizer()),
    ("RiskParity", RiskParityOptimizer()),
    ("HalfKelly", KellyOptimizer(fraction=0.5)),
]:
    strategy = Strategy(
        name=label,
        universe=StaticUniverse((SYMBOL,)),
        signals={
            "golden_cross": (Crossover(), {"fast": "sma_10", "slow": "sma_50"}),
        },
        portfolio=optimizer,
    )
    opt_results[label] = run(strategy)

compare(opt_results)

**解读**：

- `EqualWeightOptimizer` 对单标的等同于全仓，收益和风险都最大
- `RiskParityOptimizer` 和 `KellyOptimizer` 需要额外的指标列（波动率、胜率等），在缺少这些数据时可能回退到现金权重
- 实际使用中，配合多标的和合适的指标，RiskParity 和 Kelly 会展现出更好的风险调整后收益

> **关键理解**：PortfolioOptimizer 取代了旧架构中的 EntryRule、TargetValueEntryRule、FullPositionEntryRule、SizedEntryRule 和 RebalanceRule。入场、仓位大小、再平衡统一由 Optimizer 处理。

---
## 2. ExitRule — 信号退出规则

`ExitRule` 是一个基于指标交叉的退出规则：当快线低于慢线且持有仓位时，返回 `RuleResult` 指示平仓。

在新架构中，ExitRule 作为 `rules` 参数传给 `Engine.run()`。它返回 `RuleResult`（而非 Order），Engine 根据 `target_positions` 字段自动生成卖出订单。

In [ ]:
from oxq.core.types import Portfolio, Position, RuleResult
from oxq.rules import ExitRule

exit_rule = ExitRule(fast="sma_10", slow="sma_50")

# 有持仓 + 快线 < 慢线 → 返回 target_positions={AAPL: 0.0}（平仓）
row_exit = pd.Series({"close": 95.0, "sma_10": 97.0, "sma_50": 100.0})
portfolio_pos = Portfolio(
    cash=Decimal("50000"),
    positions={"AAPL": Position(symbol="AAPL", shares=100, avg_cost=Decimal("102"))},
)
result = exit_rule.evaluate("AAPL", row_exit, portfolio_pos)
print(f"快线 < 慢线 + 有持仓: {result}")
print(f"  target_positions = {result.target_positions}")
print(f"  reason = {result.reason}")

# 无持仓 → 空 RuleResult
result2 = exit_rule.evaluate("AAPL", row_exit, Portfolio(cash=Decimal("100000")))
print(f"\n快线 < 慢线 + 无持仓: {result2}")
print(f"  target_positions = {result2.target_positions}")

# 快线 > 慢线 → 空 RuleResult
row_hold = pd.Series({"close": 105.0, "sma_10": 103.0, "sma_50": 100.0})
result3 = exit_rule.evaluate("AAPL", row_hold, portfolio_pos)
print(f"\n快线 > 慢线 + 有持仓: target_positions = {result3.target_positions}")

**RuleResult 返回值解读**：

- `target_positions={symbol: 0.0}` 表示「将该标的的持仓设为 0」，即全部卖出
- `target_positions=None`（默认）表示「不做任何操作」
- Engine 收到 RuleResult 后，比较 target_positions 与当前持仓，自动生成相应的买入/卖出订单

> **注意**: `ExitRule` 使用指标列（`sma_10`, `sma_50`）判断退出条件，而非 Signal 列。
> 这是有意的设计——退出条件通常与入场条件不同。如果需要基于 Signal 退出，可以自定义 Rule。

---
## 3. Order Rules — 止损/止盈/追踪止损

Order Rules 监控持仓的盈亏状态，在达到阈值时返回 `RuleResult` 指示平仓。

在新架构中，这三种 Rule 不再生成挂单（Order），而是返回 `RuleResult`：
- 触发时：`RuleResult(target_positions={symbol: 0.0})` — 指示 Engine 平仓
- 未触发时：`RuleResult()` — 空结果，不做操作

open-xquant 提供 3 种 Order Rule：

| 规则 | 触发条件 | 行为 |
|------|----------|------|
| `StopLossRule` | close <= avg_cost * (1 - threshold) | 平仓 |
| `TakeProfitRule` | close >= avg_cost * (1 + threshold) | 平仓 |
| `TrailingStopRule` | close <= HWM * (1 - trail_pct) | 平仓 |

### 3.1 StopLossRule — 固定止损

当持有仓位时，如果当前价格跌至 `avg_cost * (1 - threshold)` 以下，返回 RuleResult 指示平仓。

例如：买入均价 200，threshold=0.05 → 止损价 190。当收盘价跌至 190 或以下时，触发。

In [ ]:
from oxq.rules import StopLossRule

stop_loss = StopLossRule(threshold=0.05)  # 5% 止损

# 有持仓，价格未跌破止损线 → 空 RuleResult
row = pd.Series({"close": 200.0})
portfolio = Portfolio(
    cash=Decimal("80000"),
    positions={"AAPL": Position(symbol="AAPL", shares=100, avg_cost=Decimal("200"))},
)
result = stop_loss.evaluate("AAPL", row, portfolio)
print(f"价格 200 (止损线 190): target_positions={result.target_positions}")

# 价格跌破止损线 → 返回 target_positions={AAPL: 0.0}
row_low = pd.Series({"close": 188.0})
result2 = stop_loss.evaluate("AAPL", row_low, portfolio)
print(f"价格 188 (<= 190):    target_positions={result2.target_positions}")
print(f"  reason: {result2.reason}")

# 无持仓 → 空 RuleResult
result3 = stop_loss.evaluate("AAPL", row_low, Portfolio(cash=Decimal("100000")))
print(f"\n无持仓: target_positions={result3.target_positions}")

### 3.2 TakeProfitRule — 固定止盈

当持有仓位时，如果当前价格涨至 `avg_cost * (1 + threshold)` 以上，返回 RuleResult 指示平仓。

例如：买入均价 200，threshold=0.15 → 止盈价 230。当收盘价涨至 230 或以上时，触发。

In [ ]:
from oxq.rules import TakeProfitRule

take_profit = TakeProfitRule(threshold=0.15)  # 15% 止盈

row = pd.Series({"close": 200.0})
portfolio = Portfolio(
    cash=Decimal("80000"),
    positions={"AAPL": Position(symbol="AAPL", shares=100, avg_cost=Decimal("200"))},
)

# 价格未达到止盈线 → 空 RuleResult
result = take_profit.evaluate("AAPL", row, portfolio)
print(f"价格 200 (止盈线 230): target_positions={result.target_positions}")

# 价格达到止盈线 → 平仓
row_high = pd.Series({"close": 235.0})
result2 = take_profit.evaluate("AAPL", row_high, portfolio)
print(f"价格 235 (>= 230):    target_positions={result2.target_positions}")
print(f"  reason: {result2.reason}")

### 3.3 TrailingStopRule — 追踪止损

跟踪价格的**高水位 (HWM)**，当价格从高水位回撤超过 `trail_pct` 时触发。

```
价格走势:  100 → 110 → 105 → 103
HWM:      100 → 110    110    110
止损线(5%):        104.5  104.5  104.5
触发?:              否     否     是 (103 <= 104.5)
```

追踪止损的优势：在趋势上涨中，止损线随价格上移，锁住更多利润；在回调中自动触发，避免利润回吐。

In [ ]:
from oxq.rules import TrailingStopRule

trailing = TrailingStopRule(trail_pct=0.05)  # 5% 追踪止损

portfolio = Portfolio(
    cash=Decimal("80000"),
    positions={"AAPL": Position(symbol="AAPL", shares=100, avg_cost=Decimal("150"))},
)

# 价格 200 → HWM = 200, 止损线 = 190
row1 = pd.Series({"close": 200.0})
result1 = trailing.evaluate("AAPL", row1, portfolio)
print(f"价格 200 (HWM=200, 止损线=190): target_positions={result1.target_positions}")

# 价格 195 → HWM = 200, 止损线 = 190, 未触发
row2 = pd.Series({"close": 195.0})
result2 = trailing.evaluate("AAPL", row2, portfolio)
print(f"价格 195 (HWM=200, 止损线=190): target_positions={result2.target_positions}")

# 价格 188 → HWM = 200, 止损线 = 190, 触发!
row3 = pd.Series({"close": 188.0})
result3 = trailing.evaluate("AAPL", row3, portfolio)
print(f"价格 188 (HWM=200, 止损线=190): target_positions={result3.target_positions}")
print(f"  reason: {result3.reason}")

### 3.4 Order Rules 回测对比

在基础的 SMA 金叉策略上，分别加入不同的 Order Rule，观察对收益和风险的影响：

In [ ]:
order_results = {}
for label, rules in [
    ("无保护(基准)", []),
    ("5%止损", [StopLossRule(threshold=0.05)]),
    ("15%止盈", [TakeProfitRule(threshold=0.15)]),
    ("5%追踪止损", [TrailingStopRule(trail_pct=0.05)]),
    ("止损+止盈", [StopLossRule(threshold=0.05), TakeProfitRule(threshold=0.15)]),
    ("追踪+止盈", [TrailingStopRule(trail_pct=0.05), TakeProfitRule(threshold=0.15)]),
]:
    order_results[label] = run(base_strategy, rules=rules)

compare(order_results)

查看「追踪+止盈」组合的交易记录：

In [ ]:
print("追踪止损+止盈 组合交易记录:")
show_trades(order_results["追踪+止盈"], max_rows=16)

**解读**：

- 所有 Rule 现在返回 `RuleResult`，Engine 根据 `target_positions` 统一执行
- 止损和止盈互不干扰——每种 Rule 独立评估
- 止损+止盈组合通常比单独使用效果更好，因为它同时控制了下行风险和锁定了上行利润
- Rules 通过 `Engine.run(rules=[...])` 传入，而非写在 Strategy 定义中

---
## 4. Risk Rules — 组合级熔断保护

Risk Rules 是策略的「安全网」——在极端行情下自动冻结交易，防止不可控的损失。

### 与 Order Rules 的关键区别

| 维度 | Order Rules (StopLoss/TakeProfit/TrailingStop) | Risk Rules |
|------|-----------------------------------------------|----------|
| `hold` 信号 | 无 | 可冻结后续所有 Rule |
| 作用范围 | 单个标的的盈亏 | 组合级别 |
| 清仓行为 | 平掉单个标的 | MaxDrawdownRisk 会清仓所有标的 |

Risk Rule 返回的 `hold=True` 时，Engine 跳过后续所有 Rule 的评估。

open-xquant 提供 2 种 Risk Rule：

| 规则 | 监控指标 | 触发行为 |
|------|----------|----------|
| `MaxDrawdownRisk` | 组合峰值到谷值回撤 | 清仓 + 冻结 |
| `DailyLossLimitRisk` | 单日亏损 | 仅冻结（不清仓） |

### 4.1 MaxDrawdownRisk — 最大回撤熔断

跟踪组合的历史峰值。当「(峰值 - 当前值) / 峰值」超过 `max_drawdown` 时：

1. 返回 `RuleResult(target_positions={symbol: 0.0}, hold=True)` — 清仓并冻结
2. Engine 收到 `hold=True` 后跳过后续所有 Rule

In [ ]:
from oxq.rules import MaxDrawdownRisk

risk = MaxDrawdownRisk(max_drawdown=0.15)  # 15% 回撤熔断

portfolio = Portfolio(
    cash=Decimal("0"),
    positions={"AAPL": Position(symbol="AAPL", shares=100, avg_cost=Decimal("150"))},
)

# 第一次评估: 组合值 = 100 * 200 = 20000，设为峰值
row_high = pd.Series({"close": 200.0})
result = risk.evaluate("AAPL", row_high, portfolio)
print(f"价格 200 (峰值): hold={result.hold}, target_positions={result.target_positions}")

# 第二次评估: 组合值 = 100 * 160 = 16000，回撤 = 4000/20000 = 20% > 15%
row_low = pd.Series({"close": 160.0})
result2 = risk.evaluate("AAPL", row_low, portfolio)
print(f"价格 160 (回撤20%): hold={result2.hold}, target_positions={result2.target_positions}")
print(f"  reason: {result2.reason}")

### 4.2 DailyLossLimitRisk — 单日亏损熔断

记录每个交易日开始时的组合价值。当日内亏损超过 `max_daily_loss` 时，返回 `hold=True` 冻结交易。

与 `MaxDrawdownRisk` 的区别：
- **不清仓** — 只冻结，不生成 target_positions
- **次日自动恢复** — 新的交易日会重置基准值
- 适合日内波动剧烈但长期趋势明确的行情

In [ ]:
from oxq.rules import DailyLossLimitRisk

daily_risk = DailyLossLimitRisk(max_daily_loss=0.03)  # 3% 单日亏损限制

portfolio = Portfolio(
    cash=Decimal("0"),
    positions={"AAPL": Position(symbol="AAPL", shares=100, avg_cost=Decimal("150"))},
)

# 日初评估: 记录基准值 = 100 * 200 = 20000
row1 = pd.Series({"close": 200.0}, name=pd.Timestamp("2024-01-02"))
result = daily_risk.evaluate("AAPL", row1, portfolio)
print(f"日初 价格 200: hold={result.hold}  (记录基准值 20000)")

# 同日价格下跌: 100 * 190 = 19000, 日损 = 1000/20000 = 5% > 3%
row2 = pd.Series({"close": 190.0}, name=pd.Timestamp("2024-01-02"))
result2 = daily_risk.evaluate("AAPL", row2, portfolio)
print(f"同日 价格 190: hold={result2.hold}  (日损 5% > 3%, 冻结交易)")
print(f"  target_positions={result2.target_positions}  (DailyLossLimitRisk 不清仓)")
print(f"  reason: {result2.reason}")

### 4.3 Risk Rules 回测对比

In [ ]:
risk_results = {}
for label, rules in [
    ("无保护(基准)", []),
    ("15%最大回撤", [MaxDrawdownRisk(max_drawdown=0.15)]),
    ("3%单日亏损", [DailyLossLimitRisk(max_daily_loss=0.03)]),
    ("双重保护", [
        MaxDrawdownRisk(max_drawdown=0.15),
        DailyLossLimitRisk(max_daily_loss=0.03),
    ]),
]:
    risk_results[label] = run(base_strategy, rules=rules)

compare(risk_results)

**解读**：

- Risk Rules 是「保险」——在行情温和时不影响策略表现，但在极端行情下能有效控制损失
- `DailyLossLimitRisk` 的冻结是日内的——次日自动恢复交易
- 两者可叠加使用：任一返回 `hold=True` 即冻结后续所有 Rule
- 在新架构中，Risk Rules 通过 `rules` 参数传入，Engine 先评估所有 Rule，遇到 `hold=True` 时跳过后续

---
## 5. 交易成本 — 手续费与滑点

交易成本不是 Rule，但它直接影响策略的最终表现。SimBroker 通过两个 Protocol 接口模拟交易成本：

| 模型 | 作用 | 公式 |
|------|------|------|
| `PercentageFee` | 手续费 | `max(fill_price * shares * rate, min_fee)` |
| `PercentageSlippage` | 滑点 | BUY: `price * (1 + rate)`，SELL: `price * (1 - rate)` |

**执行顺序**：先计算滑点调整后的成交价，再按调整后价格计算手续费。

In [ ]:
from oxq.trade import PercentageFee, PercentageSlippage

# 查看手续费模型的默认参数
fee = PercentageFee()  # rate=0.1%, min_fee=5
print(f"PercentageFee: rate={fee.rate}, min_fee={fee.min_fee}")

# 查看滑点模型的默认参数
slip = PercentageSlippage()  # rate=0.1%
print(f"PercentageSlippage: rate={slip.rate}")

# 手动计算: 买 100 股 AAPL @200
from oxq.core.types import Order
order = Order(symbol="AAPL", side="BUY", shares=100)

raw_price = Decimal("200")
slipped_price = slip.adjust(order, raw_price)
fee_amount = fee.calculate(order, slipped_price)

print(f"\n手动计算: BUY 100 股 @200")
print(f"  滑点后价格: {slipped_price}  (200 × 1.001)")
print(f"  手续费:     {fee_amount}  ({slipped_price} × 100 × 0.001)")
print(f"  总成本:     {slipped_price * 100 + fee_amount}")

### 交易成本对策略的影响

同一个策略，在不同交易成本下的表现差异：

In [ ]:
from oxq.trade import PercentageFee, PercentageSlippage

cost_configs = {
    "无成本(基准)": {},
    "0.1%手续费": {
        "fee_model": PercentageFee(rate=Decimal("0.001"), min_fee=Decimal("5")),
    },
    "0.1%滑点": {
        "slippage_model": PercentageSlippage(rate=Decimal("0.001")),
    },
    "费用+滑点": {
        "fee_model": PercentageFee(rate=Decimal("0.001"), min_fee=Decimal("5")),
        "slippage_model": PercentageSlippage(rate=Decimal("0.001")),
    },
    "高成本": {
        "fee_model": PercentageFee(rate=Decimal("0.003"), min_fee=Decimal("10")),
        "slippage_model": PercentageSlippage(rate=Decimal("0.005")),
    },
}

cost_results = {}
for label, kwargs in cost_configs.items():
    cost_results[label] = run(base_strategy, **kwargs)

compare(cost_results, extra_rows=[
    ("总手续费", lambda r: f"{sum(f.fee for f in r.trades):.0f}"),
])

**解读**：

- 交易成本是回测与实盘表现差异的主要来源之一
- 对低频策略（本例 ~13 笔交易），0.1% 的费率影响约 0.3%~0.5% 收益率
- 对高频策略，影响会成倍放大
- `min_fee` 对小额交易的影响尤其大——100 股 × 200 = 20,000 的交易，0.1% 费率只有 20 元，但 min_fee=5 兜底
- 在策略开发后期，加入交易成本是评估可行性的必要步骤

---
## 6. 完整策略 — Strategy + Rules 协同工作

将 PortfolioOptimizer 和所有 Rule 组合成一个完整的、具备风控能力的策略：

```
Strategy 定义:
  - name, universe, signals
  - portfolio = EqualWeightOptimizer()  ← 自动入场和仓位管理

Engine.run(rules=[...]):
  1. MaxDrawdownRisk     → 检查回撤，超过 15% 则清仓 + 冻结
  2. StopLossRule        → 价格跌破 avg_cost * 0.95 → 平仓
  3. TakeProfitRule      → 价格涨破 avg_cost * 1.20 → 平仓
  4. ExitRule            → 死叉时平仓
```

In [ ]:
full_strategy = Strategy(
    name="full_rules_demo",
    hypothesis=(
        "SMA 金叉信号 + EqualWeight 仓位管理，配合止损止盈和回撤熔断，"
        "在控制下行风险的同时捕获趋势收益"
    ),
    objectives={
        "total_return": {"min": 0.0},
        "sharpe_ratio": {"min": 0.3},
        "max_drawdown": {"max": -0.15},
    },
    universe=StaticUniverse((SYMBOL,)),
    signals={
        "golden_cross": (Crossover(), {"fast": "sma_10", "slow": "sma_50"}),
    },
    portfolio=EqualWeightOptimizer(),
)

full_rules = [
    MaxDrawdownRisk(max_drawdown=0.15),
    StopLossRule(threshold=0.05),
    TakeProfitRule(threshold=0.20),
    ExitRule(fast="sma_10", slow="sma_50"),
]

result = run(
    full_strategy,
    rules=full_rules,
    fee_model=PercentageFee(rate=Decimal("0.001"), min_fee=Decimal("5")),
    slippage_model=PercentageSlippage(rate=Decimal("0.001")),
)

print(f"总收益率:   {result.total_return():.2%}")
print(f"Sharpe Ratio: {result.sharpe_ratio():.2f}")
print(f"最大回撤:   {result.max_drawdown():.2%}")
print(f"交易次数:   {len(result.trades)}")
print(f"期末资产:   {result.equity_curve[-1][1]:,.0f}")
print(f"总手续费:   {sum(f.fee for f in result.trades):.2f}")

### 目标检查

In [ ]:
metrics = {
    "total_return": result.total_return(),
    "sharpe_ratio": result.sharpe_ratio(),
    "max_drawdown": result.max_drawdown(),
}

print("目标检查:")
for metric_name, bounds in full_strategy.objectives.items():
    actual = metrics[metric_name]
    passed = True
    if "min" in bounds:
        passed = passed and actual >= bounds["min"]
    if "max" in bounds:
        passed = passed and actual <= bounds["max"]
    status = "PASS" if passed else "FAIL"
    print(f"  {metric_name:<16} = {actual:>8.4f}  [{status}]")

### 完整交易记录

In [ ]:
show_trades(result, max_rows=20)

---
## 小结

### 新架构总览

```
Strategy(name, universe, signals, portfolio=PortfolioOptimizer)
    ↓
Engine.run(strategy, market, broker, rules=[...], start, end)
```

- **入场与仓位** — 由 PortfolioOptimizer 自动处理（EqualWeight / RiskParity / Kelly）
- **风控与退出** — 由 Rules 列表处理，传给 `Engine.run()`

### PortfolioOptimizer 速查表

| Optimizer | 权重逻辑 | 适用场景 |
|-----------|----------|----------|
| `EqualWeightOptimizer` | 等权分配 | 分散投资、快速验证 |
| `RiskParityOptimizer` | 波动率倒数加权 | 风险均衡配置 |
| `KellyOptimizer` | Kelly 公式最优仓位 | 有历史胜率数据 |

### Rule 速查表

| Rule | 返回值 | 职责 |
|------|--------|------|
| `ExitRule` | `RuleResult(target_positions={sym: 0.0})` | 死叉时平仓 |
| `StopLossRule` | `RuleResult(target_positions={sym: 0.0})` | 固定止损 |
| `TakeProfitRule` | `RuleResult(target_positions={sym: 0.0})` | 固定止盈 |
| `TrailingStopRule` | `RuleResult(target_positions={sym: 0.0})` | 追踪止损 |
| `MaxDrawdownRisk` | `RuleResult(target_positions={sym: 0.0}, hold=True)` | 组合回撤熔断 + 清仓 |
| `DailyLossLimitRisk` | `RuleResult(hold=True)` | 单日亏损熔断（不清仓） |

### RuleResult 字段

| 字段 | 类型 | 说明 |
|------|------|------|
| `weights` | `dict[str, float] \| None` | 目标权重覆盖 |
| `constraints` | `dict[str, Constraint] \| None` | 单标的交易约束 |
| `target_positions` | `dict[str, float] \| None` | 绝对目标持仓 |
| `hold` | `bool` | 冻结交易（跳过后续 Rule） |
| `reason` | `str` | 人类可读的触发原因 |

### 交易成本

| 模型 | 实现 | 作用 |
|------|------|------|
| `FeeModel` | `PercentageFee` | 按交易额比例收费，有最低收费 |
| `SlippageModel` | `PercentageSlippage` | 模拟市场冲击（买贵卖便宜） |

### 已移除的旧 API

以下 API 在新架构中已被移除：

| 旧 API | 替代方案 |
|--------|---------|
| `EntryRule` | PortfolioOptimizer 自动入场 |
| `TargetValueEntryRule` | PortfolioOptimizer 权重分配 |
| `FullPositionEntryRule` | EqualWeightOptimizer (单标的=全仓) |
| `SizedEntryRule` | PortfolioOptimizer + Rule constraints |
| `RebalanceRule` | PortfolioOptimizer 每 bar 自动再平衡 |
| `clip_to_max_position` | 已移除 |
| `clip_to_pct_equity` | 已移除 |
| `os_equal_weight` 等 | PortfolioOptimizer |

### 核心设计原则

- **Strategy 纯函数** — Strategy 只定义 "what"（信号 + 权重），不关心 "how"（订单执行）
- **Rules 外置** — Rules 通过 `Engine.run(rules=[...])` 传入，可灵活组合
- **统一返回值** — 所有 Rule 返回 `RuleResult`，Engine 统一处理
- **hold 机制** — Risk Rules 的 `hold=True` 冻结后续 Rule 评估
- **Protocol 接口** — FeeModel、SlippageModel、PortfolioOptimizer 均可自定义实现

### 相关教程

- `engine_module.ipynb` — Engine 管道详解
- `signal_comparison.ipynb` — Signal 模块详解（Crossover 等）